====================================================================================================== <br>
This notebook merges InTEM inner and outer regions and generates a basis functions netcdf output file. <br>
====================================================================================================== <br>

Authors: Helene De Longueville and Alexandre Danjou <br>
Project: PARIS <br>
Year: 2024

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
from datetime import datetime

import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

## A. Inputs

In [ ]:
year = '2022'
domain = 'EUROPE'

fn1 = f'/user/work/qq24644/my_paris/InTEM_basis_functions/MHT2J1M5_N_pfc218_OBUSEXL_4h_gsize_{year}0101_{year}1231_F.txt' # Inner regions filename
fn2 = '/user/work/qq24644/my_paris/InTEM_basis_functions/intem_region_definition.nc' # Outer regions filename

output_fn = f'/group/chemistry/acrg/LPDM/basis_functions/EUROPE/intem_pfc218_{year}_EUROPE_PARIS.nc' # Output filename

## B. Extract Inner Regions

In [ ]:
# get data
inner = pd.read_csv(fn1, sep='\s+', skiprows=12, usecols=(3, 4, 5), header=None)
col = (pd.read_csv(fn1, sep='\s+',skiprows=11, nrows=1, usecols=(3, 4, 5), header=None)).iloc[0].tolist()

inner.columns = col

# change format
inner = inner.rename(columns={'Lon': 'lon', 'Lat': 'lat', 'Code': 'basis'}).set_index(['lat','lon'])
inner = inner.to_xarray()

In [ ]:
# check data plot
vals = np.linspace(0,1,inner.basis.max().item()+1)
np.random.shuffle(vals)
cmap = plt.cm.colors.ListedColormap(plt.cm.jet(vals))

fig1, ax1 = plt.subplots(subplot_kw={'projection': ccrs.PlateCarree()}, figsize=(5,5))
inner.basis.plot(ax=ax1, transform=ccrs.PlateCarree(), cbar_kwargs={"shrink":0.77}, cmap=cmap)
ax1.add_feature(cfeature.COASTLINE, linewidth=0.5, edgecolor='black')
ax1.add_feature(cfeature.BORDERS, linewidth=0.5, edgecolor='black')
gl1 = ax1.gridlines(draw_labels=True, linewidth=0.5, color='gray', alpha=0.6, linestyle='--')
gl1.right_labels = False
gl1.bottom_labels = False

## C. Extract Outer Regions

In [ ]:
# get data
with xr.open_dataset(fn2) as outer:   
    outer = outer

# change format
outer = outer.rename({'region': 'basis'})

In [ ]:
# check data plot
vals = np.linspace(0,1,outer.basis.max().item()+1)
np.random.shuffle(vals)
cmap = plt.cm.colors.ListedColormap(plt.cm.jet(vals))

fig2, ax2 = plt.subplots(subplot_kw={'projection': ccrs.PlateCarree()}, figsize=(7,7))
outer.basis.plot(ax=ax2, transform=ccrs.PlateCarree(), cbar_kwargs={"shrink":0.41}, cmap=cmap)
ax2.add_feature(cfeature.COASTLINE, linewidth=0.5, edgecolor='black')
ax2.add_feature(cfeature.BORDERS, linewidth=0.5, edgecolor='black')
gl2 = ax2.gridlines(draw_labels=True, linewidth=0.5, color='gray', alpha=0.6, linestyle='--')
gl2.right_labels = False
gl2.bottom_labels = False

## D. Merge Inner and Outer Regions

In [ ]:
merged = outer
merged.attrs = {}

inner_mask_y, inner_mask_x = np.where(outer.basis==6)

for i in range(len(inner_mask_y)):
    merged.basis.values[inner_mask_y[i], inner_mask_x[i]] = inner.basis.values.ravel()[i]+5

if merged.basis.values.min() == 0:
    merged = merged+1 # must start from 1

In [ ]:
# check data plot
vals = np.linspace(0,1,merged.basis.max().item()+1)
np.random.shuffle(vals)
cmap = plt.cm.colors.ListedColormap(plt.cm.jet(vals))

fig3, ax3 = plt.subplots(subplot_kw={'projection': ccrs.PlateCarree()}, figsize=(7,7))
merged.basis.plot(ax=ax3, transform=ccrs.PlateCarree(), cbar_kwargs={"shrink":0.41}, cmap=cmap)
ax3.add_feature(cfeature.COASTLINE, linewidth=0.5, edgecolor='black')
ax3.add_feature(cfeature.BORDERS, linewidth=0.5, edgecolor='black')
gl3 = ax3.gridlines(draw_labels=True, linewidth=0.5, color='gray', alpha=0.6, linestyle='--')
gl3.right_labels = False
gl3.bottom_labels = False

## E. Fix geographic coordinates

In [ ]:
from openghg.util import find_domain

[lat_ref, lon_ref] = find_domain(domain)

merged = merged.assign_coords({'lat': lat_ref}).assign_coords({'lon': lon_ref})

## F. Assign attributes

In [ ]:
merged.lat.attrs['long_name'] = 'latitude'
merged.lat.attrs['units'] = 'degrees_north'
merged.lon.attrs['long_name'] = 'longitude'
merged.lon.attrs['units'] = 'degrees_east'
merged.basis.attrs['long_name'] = 'InTEM inner and outer regions definition'

merged.attrs['description'] = 'PARIS basis functions from InTEM (inner and outer regions)'
merged.attrs['created_from'] = f'merge of {fn1} and {fn2}'
merged.attrs['created_by'] = 'qq24644'
merged.attrs['creation_on'] = str(datetime.now())

merged = merged.expand_dims(dim={"time": 1}, axis=2).assign_coords({"time": ("time", [np.datetime64(year+'-07-01')])})

## G. Save output

In [ ]:
merged.to_netcdf(output_fn, mode="w")